# Compana AI/ML Training Overview

Notebook ini dibuat untuk memperlihatkan bagian AI/ML dari engine Compana secara eksplisit: data yang dipakai, data scraping pendukung, training baseline model, evaluasi, dan contoh inference.

Catatan: engine MVP utama masih rule-based agar deterministik dan mudah diuji. Model ML di notebook ini adalah baseline untuk `problem_category` classification dari `pretext_text`.

## 1. Setup

Jalankan notebook dari root repo. Jika memakai Jupyter, pastikan dependencies dari `requirements.txt` sudah terpasang.

In [38]:
from pathlib import Path
import json
import sys
import os

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_colwidth", 120)
print(PROJECT_ROOT)

D:\NURUL\Dicoding\CapstoneProject\ModelAI-ML


## 2. Load Data

Ada dua jenis data yang dilihat di notebook ini:

- Synthetic labeled data untuk training classifier `problem_category`.
- Scraped job posting data untuk mendukung role/skill/task knowledge base Engine 5-8.

In [39]:
TRAIN_DATA = PROJECT_ROOT / "data" / "labels" / "compana_synthetic_expanded_v1.csv"
SCRAPED_JOBS = PROJECT_ROOT / "ai_engines_5_8" / "data" / "processed" / "job_postings_flat.csv"
SCRAPED_ROLE_SKILLS = PROJECT_ROOT / "ai_engines_5_8" / "data" / "processed" / "role_skill_mapping.csv"

train_df = pd.read_csv(TRAIN_DATA)
jobs_df = pd.read_csv(SCRAPED_JOBS) if SCRAPED_JOBS.exists() else pd.DataFrame()
scraped_role_skill_df = pd.read_csv(SCRAPED_ROLE_SKILLS) if SCRAPED_ROLE_SKILLS.exists() else pd.DataFrame()

print("training rows:", len(train_df))
print("scraped job rows:", len(jobs_df))
print("scraped role-skill rows:", len(scraped_role_skill_df))
train_df.head()

training rows: 480
scraped job rows: 11
scraped role-skill rows: 28


,pretext_text,target_role,problem_category,current_level,blocker_type
0,"Ada banyak framework frontend, saya tidak tahu yang mana untuk frontend_developer.",frontend_developer,frontend_task,beginner,none
1,"Saya baru mulai frontend_developer, tidak tahu harus mulai dari mana.",frontend_developer,frontend_task,beginner,none
2,Saya tidak mengerti JavaScript fundamentals untuk frontend_developer.,frontend_developer,frontend_task,beginner,none
3,Bagaimana cara memulai karir sebagai frontend_developer? Saya sangat bingung.,frontend_developer,frontend_task,beginner,none
4,"Ada banyak framework frontend, saya tidak tahu yang mana untuk frontend_developer.",frontend_developer,frontend_task,beginner,none


In [40]:
label_summary = (
    train_df.groupby(["problem_category", "target_role", "current_level"])
    .size()
    .reset_index(name="rows")
    .sort_values(["problem_category", "target_role", "current_level"])
)
label_summary.head(20)

,problem_category,target_role,current_level,rows
0,backend_task,backend_developer,advanced,20
1,backend_task,backend_developer,basic,20
2,backend_task,backend_developer,beginner,20
3,backend_task,backend_developer,intermediate,20
4,backend_task,data_analyst,advanced,20
5,backend_task,data_analyst,basic,20
6,backend_task,data_analyst,beginner,20
7,backend_task,data_analyst,intermediate,20
8,backend_task,frontend_developer,advanced,20
9,backend_task,frontend_developer,basic,20


In [41]:
if not jobs_df.empty:
    display(jobs_df[["title", "company", "detected_role", "detected_keywords", "source_url"]])
else:
    print("Scraped jobs belum tersedia. Jalankan ai_engines_5_8/src/scraping/scrape_jobposting_jsonld.py dulu.")

,title,company,detected_role,detected_keywords,source_url
0,Frontend Engineer,Clarify,frontend_developer,frontend;react,https://jobs.ashbyhq.com/clarify/3051d6cb-b1fe-40cf-bc10-61a521ca9d08
1,"Frontend Engineer, ChatGPT Engineering",OpenAI,frontend_developer,frontend;react,https://jobs.ashbyhq.com/openai/5bde9af5-df78-460e-ae9c-5c49ac778640
2,Backend Engineer,Boam AI,backend_developer,backend;api;python,https://jobs.ashbyhq.com/boam/a7c8e2a7-ef4e-4768-a16d-ed258e57d90d/
3,Security Engineer,Hive,cyber_security_analyst,security;cyber,https://jobs.lever.co/hive/b1150ab5-15dd-45f5-8ad3-a4d562d800d6
4,Frontend Engineer,Procreate,frontend_developer,frontend;react;html;css;vue,https://jobs.lever.co/procreate/38e3f80e-370b-4eee-a69d-0ce7a60afb6a
5,Staff Security Engineer (Blue Team),Olo,cyber_security_analyst,security;siem;cyber,https://jobs.lever.co/olo/b19e4037-f645-47d0-bc29-10a714fa19e7
6,Frontend Engineer - Growth team,Insify,frontend_developer,frontend;react;html;css,https://jobs.lever.co/insify/bf641b5a-82f8-4b79-9c3b-c7d381a42d89
7,Security Engineer - Product Security,Aircall,cyber_security_analyst,security,https://jobs.lever.co/aircall/2313b30a-1136-4db9-bc1d-26337a92823d
8,"Senior Application Security Engineer, AI &amp; Product Security",Artera,cyber_security_analyst,security,https://jobs.lever.co/artera-2/c77979d2-dd2a-49c3-9945-1083fc6a08a9
9,Senior Data Analyst,"CIM Group, LP",data_analyst,data analyst;analytics;analyst;tableau;power bi;sql,https://jobs.lever.co/cimgroup/c1570490-e240-4f03-97ff-3a91a7433a89


In [42]:
if not scraped_role_skill_df.empty:
    display(scraped_role_skill_df.groupby(["peran", "keahlian", "priority"]).size().reset_index(name="evidence_count"))
else:
    print("Scraped role-skill mapping belum tersedia.")

,peran,keahlian,priority,evidence_count
0,backend_developer,Java,medium,1
1,backend_developer,Python,medium,4
2,backend_developer,SQL,medium,2
3,cyber_security_analyst,Linux,medium,1
4,cyber_security_analyst,SIEM,medium,1
5,data_analyst,Power BI,medium,2
6,data_analyst,SQL,medium,2
7,data_analyst,Tableau,medium,2
8,frontend_developer,CSS,medium,2
9,frontend_developer,HTML,medium,2


## 3. ML Problem Definition

Target ML baseline:

- Input: `pretext_text`, yaitu teks bebas user.
- Output: `problem_category`.
- Model: TF-IDF text features + Logistic Regression.

Baseline ini membantu melihat apakah model statistik sederhana bisa mengenali kategori masalah dari variasi kalimat user.

In [43]:
TEXT_COL = "pretext_text"
LABEL_COL = "problem_category"
RANDOM_STATE = 42

ml_df = train_df[[TEXT_COL, LABEL_COL]].dropna().copy()
ml_df[TEXT_COL] = ml_df[TEXT_COL].astype(str)
ml_df[LABEL_COL] = ml_df[LABEL_COL].astype(str)

labels = sorted(ml_df[LABEL_COL].unique())
labels

['backend_task', 'frontend_task']

In [44]:
train_part, temp_part = train_test_split(
    ml_df,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=ml_df[LABEL_COL],
)
dev_part, test_part = train_test_split(
    temp_part,
    test_size=2/3,
    random_state=RANDOM_STATE,
    stratify=temp_part[LABEL_COL],
)

split_summary = pd.DataFrame(
    {
        "split": ["train", "dev", "test"],
        "rows": [len(train_part), len(dev_part), len(test_part)],
    }
)
split_summary

,split,rows
0,train,336
1,dev,48
2,test,96


## 4. Train Model

In [45]:
model = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
        ("clf", LogisticRegression(max_iter=300, random_state=RANDOM_STATE)),
    ]
)

model.fit(train_part[TEXT_COL], train_part[LABEL_COL])
model

,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


## 5. Evaluate Model

In [46]:
dev_pred = model.predict(dev_part[TEXT_COL])
test_pred = model.predict(test_part[TEXT_COL])

metrics = {
    "dev_accuracy": accuracy_score(dev_part[LABEL_COL], dev_pred),
    "dev_macro_f1": f1_score(dev_part[LABEL_COL], dev_pred, average="macro"),
    "test_accuracy": accuracy_score(test_part[LABEL_COL], test_pred),
    "test_macro_f1": f1_score(test_part[LABEL_COL], test_pred, average="macro"),
}
metrics

{'dev_accuracy': 0.9583333333333334,
 'dev_macro_f1': 0.9582608695652174,
 'test_accuracy': 0.96875,
 'test_macro_f1': 0.9687194525904204}

In [47]:
report_df = pd.DataFrame(classification_report(test_part[LABEL_COL], test_pred, output_dict=True, zero_division=0)).T
report_df

,precision,recall,f1-score,support
backend_task,1.000000,0.93750,0.967742,48.00000
frontend_task,0.941176,1.00000,0.969697,48.00000
accuracy,0.968750,0.96875,0.968750,0.96875
macro avg,0.970588,0.96875,0.968719,96.00000
weighted avg,0.970588,0.96875,0.968719,96.00000


In [48]:
cm = confusion_matrix(test_part[LABEL_COL], test_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in labels], columns=[f"pred_{x}" for x in labels])
cm_df

,pred_backend_task,pred_frontend_task
true_backend_task,45,3
true_frontend_task,0,48


In [49]:
predictions_df = test_part.copy()
predictions_df["ml_prediction"] = test_pred
predictions_df.head(20)

,pretext_text,problem_category,ml_prediction
177,Saya takut tidak cukup smart untuk UI/UX di data_analyst.,frontend_task,frontend_task
332,Saya mengalami kesulitan memahami database concepts untuk backend_developer.,backend_task,backend_task
431,Saya mengalami kesulitan memahami database concepts untuk data_analyst.,backend_task,backend_task
389,Saya stuck dengan server-side concepts sebagai backend_developer.,backend_task,backend_task
146,"Saya baru mulai backend_developer, tidak tahu harus mulai dari mana.",frontend_task,frontend_task
400,Saya tidak mengerti REST API fundamentals untuk data_analyst.,backend_task,backend_task
233,"Ada banyak framework frontend, saya tidak tahu yang mana untuk data_analyst.",frontend_task,frontend_task
411,Saya tidak mengerti REST API fundamentals untuk data_analyst.,backend_task,backend_task
342,Saya bingung apakah harus focus di DevOps atau coding sebagai backend_developer.,backend_task,backend_task
38,Saya stuck di awal belajar React sebagai frontend_developer.,frontend_task,frontend_task


## 6. Save Artifacts

Output notebook ini disimpan ke folder `evaluation/outputs/notebook_problem_category_baseline/`. Folder `evaluation/outputs/` di-ignore karena berisi artifact generated.

In [50]:
OUT_DIR = PROJECT_ROOT / "evaluation" / "outputs" / "notebook_problem_category_baseline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(model, OUT_DIR / "problem_category_logreg_notebook.joblib")
report_df.to_csv(OUT_DIR / "classification_report.csv")
cm_df.to_csv(OUT_DIR / "confusion_matrix.csv")
predictions_df.to_csv(OUT_DIR / "test_predictions.csv", index=False)
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

sorted(p.name for p in OUT_DIR.iterdir())

['classification_report.csv',
 'confusion_matrix.csv',
 'metrics.json',
 'problem_category_logreg_notebook.joblib',
 'test_predictions.csv']

## 7. Inference Examples

Cell ini memperlihatkan bagaimana model yang sudah ditrain memprediksi input user baru.

In [51]:
examples = pd.DataFrame(
    {
        "pretext_text": [
            "Saya ingin jadi frontend developer tapi bingung harus mulai dari HTML atau React.",
            "Saya sudah belajar backend tapi masih kesulitan membuat API yang rapi.",
            "Saya ingin jadi data analyst tapi belum paham SQL dan dashboard.",
        ]
    }
)
examples["predicted_problem_category"] = model.predict(examples["pretext_text"])
examples

,pretext_text,predicted_problem_category
0,Saya ingin jadi frontend developer tapi bingung harus mulai dari HTML atau React.,frontend_task
1,Saya sudah belajar backend tapi masih kesulitan membuat API yang rapi.,backend_task
2,Saya ingin jadi data analyst tapi belum paham SQL dan dashboard.,backend_task


## 8. Interpretation

- Model ini adalah baseline sederhana, bukan final production model.
- Dataset training masih synthetic, jadi metrik tinggi belum otomatis berarti siap produksi.
- Scraped job posting digunakan sebagai evidence untuk role-skill/task enrichment, bukan sebagai label supervised training utama.
- Langkah berikutnya: kumpulkan real user interaction, validasi label manual, lalu retrain dan bandingkan dengan rule-based fallback.